# 09 - Fidelity-Aware Selector and Robust Constraint Handling

## Milestone objective

Refine the existing Component 2 fixed-budget review-evidence selector by adding:

- explicit opinion-fidelity control using Jensen-Shannon Divergence (JSD),
- a bounded deterministic repair stage,
- preservation of the existing minority-reserve mechanism,
- explicit handling of infeasible and edge-case conditions,
- reproducibility checks,
- fidelity-threshold sensitivity analysis,
- current-versus-refined selector comparison.

## Important correction in this version

Fidelity repair must not accidentally remove reviews that were selected during the
minority-reserve phase of the existing Full Proposed selector.

Therefore, minority-reserved reviews are recovered from the existing selector audit
and locked during fidelity repair. Only non-reserved reviews can be replaced.

`no_minority_required` is used only when the product truly has no qualified supported-minority
evidence. If supported-minority evidence exists but the reserve selection cannot be recovered,
the selector returns an explicit failure instead of silently ignoring the minority constraint.

## Research boundary

Real Component 1 outputs are not yet available.

The current C1 aspect-sentiment evidence is a synthetic integration-test fixture.
Therefore, C1-derived minority, coverage, and fidelity values in this milestone are
prototype diagnostics and must not be presented as final research findings.

The fidelity threshold is also provisional and must be reconsidered after real C1
integration and broader smartphone/laptop experiments.


## 1. Imports and project paths


In [20]:
from pathlib import Path
import ast
import itertools
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from scipy.spatial.distance import jensenshannon
from sklearn.metrics.pairwise import cosine_similarity


warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


def find_component_root(start: Path) -> Path:
    start = start.resolve()

    for base in [start, *start.parents]:
        if base.name == "component_2":
            return base

        candidate = base / "component_2"

        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate component_2."
    )


COMPONENT_ROOT = find_component_root(Path.cwd())

RESULTS_ROOT = (
    COMPONENT_ROOT
    / "results"
)

OUTPUT_DIR = (
    RESULTS_ROOT
    / "selector_refinement"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("Component root:", COMPONENT_ROOT)
print("Results root:", RESULTS_ROOT)
print("Output directory:", OUTPUT_DIR)


Component root: D:\Research\Repository\J26-DS-349\component_2
Results root: D:\Research\Repository\J26-DS-349\component_2\results
Output directory: D:\Research\Repository\J26-DS-349\component_2\results\selector_refinement


## 2. Schema and file-discovery helpers

The notebook discovers previous C2 outputs using both schema and filename hints.


In [21]:
PRODUCT_ALIASES = [
    "parent_asin",
    "product_id",
    "product",
    "asin_parent",
]

REVIEW_ALIASES = [
    "review_id",
    "stable_review_id",
]

METHOD_ALIASES = [
    "method",
    "selector",
    "selector_name",
    "selection_method",
    "strategy",
]

PHASE_ALIASES = [
    "selection_phase",
    "phase",
    "selector_phase",
]

RANK_ALIASES = [
    "selection_rank",
    "rank",
]

USEFULNESS_ALIASES = [
    "usefulness_norm",
    "predicted_usefulness_norm",
    "normalized_usefulness",
    "predicted_usefulness",
    "usefulness_score",
    "usefulness",
]

REPRESENTATIVENESS_ALIASES = [
    "representativeness_norm",
    "semantic_representativeness_norm",
    "normalized_representativeness",
    "representativeness",
    "semantic_representativeness",
]

MINORITY_GROUP_ALIASES = [
    "supported_minority_groups",
    "qualified_minority_groups",
    "minority_groups",
]

ASPECT_ALIASES = [
    "aspect_category",
    "aspect",
    "aspect_label",
]

SENTIMENT_ALIASES = [
    "sentiment",
    "sentiment_label",
    "polarity",
]

CONFIDENCE_ALIASES = [
    "confidence",
    "prediction_confidence",
    "aspect_sentiment_confidence",
    "avg_confidence",
    "mean_confidence",
]

SUPPORTED_FLAG_ALIASES = [
    "is_supported_minority",
    "supported_minority",
    "is_qualified_minority",
    "qualified_minority",
    "is_supported",
]

SUPPORT_ALIASES = [
    "support",
    "support_count",
    "group_support",
    "n_group",
    "n_g",
]

PREVALENCE_ALIASES = [
    "prevalence",
    "minority_prevalence",
    "group_prevalence",
    "share",
    "p_g",
]

DIRECT_TARGET_GROUP_ALIASES = [
    "priority_minority_group",
    "target_minority_group",
    "reserved_for_group",
    "minority_target_group",
    "priority_group",
    "target_group",
]


def normalize_name(value) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def resolve_column(
    columns,
    aliases,
    required=True,
):
    normalized = {
        normalize_name(column): column
        for column in columns
    }

    for alias in aliases:
        key = normalize_name(alias)

        if key in normalized:
            return normalized[key]

    if required:
        raise KeyError(
            f"Could not resolve any of {aliases}. "
            f"Available columns: {list(columns)}"
        )

    return None


def read_table(path: Path):
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(
        f"Unsupported table format: {path}"
    )


def all_result_tables():
    paths = list(
        RESULTS_ROOT.rglob("*.csv")
    )

    paths += list(
        RESULTS_ROOT.rglob("*.parquet")
    )

    return sorted(
        set(paths)
    )


def discover_table(
    required_alias_groups,
    preferred_terms=(),
    exclude_terms=(),
    prefer_largest=False,
):
    candidates = []

    for path in all_result_tables():

        path_text = str(path).lower()

        if any(
            term.lower() in path_text
            for term in exclude_terms
        ):
            continue

        try:
            frame = read_table(path)
        except Exception:
            continue

        valid = True

        for alias_group in required_alias_groups:
            try:
                resolve_column(
                    frame.columns,
                    alias_group,
                    required=True,
                )

            except KeyError:
                valid = False
                break

        if not valid:
            continue

        preference_score = sum(
            1
            for term in preferred_terms
            if term.lower() in path_text
        )

        size_score = (
            len(frame)
            if prefer_largest
            else 0
        )

        candidates.append(
            (
                preference_score,
                size_score,
                -len(str(path)),
                path,
                frame,
            )
        )

    if not candidates:
        raise FileNotFoundError(
            "No compatible table discovered for "
            f"{required_alias_groups}"
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
            item[2],
        ),
        reverse=True,
    )

    _, _, _, path, frame = (
        candidates[0]
    )

    return path, frame


def relative_path(path):
    if path is None:
        return None

    try:
        return str(
            Path(path).relative_to(
                COMPONENT_ROOT
            )
        )

    except Exception:
        return str(path)


def parse_bool(value):
    if isinstance(value, bool):
        return value

    if value is None:
        return False

    if (
        isinstance(value, float)
        and np.isnan(value)
    ):
        return False

    text = (
        str(value)
        .strip()
        .lower()
    )

    return text in {
        "true",
        "1",
        "yes",
        "y",
        "supported",
        "qualified",
    }


def canonical_group(
    aspect,
    sentiment=None,
):
    if sentiment is not None:
        return (
            f"{str(aspect).strip()}"
            f"::{str(sentiment).strip()}"
        )

    value = aspect

    if isinstance(
        value,
        (list, tuple),
    ):
        if len(value) >= 2:
            return (
                f"{str(value[0]).strip()}"
                f"::{str(value[1]).strip()}"
            )

        if len(value) == 1:
            return str(
                value[0]
            ).strip()

    return str(value).strip()


def parse_group_collection(value):
    if value is None:
        return set()

    if (
        isinstance(value, float)
        and np.isnan(value)
    ):
        return set()

    if isinstance(value, set):
        return {
            canonical_group(item)
            for item in value
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        if (
            len(value) == 2
            and not isinstance(
                value[0],
                (list, tuple, set),
            )
        ):
            return {
                canonical_group(value)
            }

        return {
            canonical_group(item)
            for item in value
        }

    text = str(value).strip()

    if text.lower() in {
        "",
        "none",
        "nan",
        "[]",
        "{}",
    }:
        return set()

    try:
        parsed = ast.literal_eval(
            text
        )

        if parsed != text:
            return parse_group_collection(
                parsed
            )

    except Exception:
        pass

    if ";" in text:
        return {
            item.strip()
            for item in text.split(";")
            if item.strip()
        }

    return {text}


## 3. Load current C2 selector inputs


In [22]:
selector_path, selector_raw = discover_table(
    required_alias_groups=[
        REVIEW_ALIASES,
        PRODUCT_ALIASES,
        USEFULNESS_ALIASES,
        REPRESENTATIVENESS_ALIASES,
    ],
    preferred_terms=(
        "selector",
        "ready",
    ),
    exclude_terms=(
        "selected_reviews",
        "baseline_selected",
        "selector_refinement",
        "pp1_evidence",
    ),
    prefer_largest=True,
)


baseline_path, baseline_raw = discover_table(
    required_alias_groups=[
        REVIEW_ALIASES,
        PRODUCT_ALIASES,
        METHOD_ALIASES,
    ],
    preferred_terms=(
        "baseline",
        "selected",
        "reviews",
    ),
    exclude_terms=(
        "selector_refinement",
        "pp1_evidence",
    ),
    prefer_largest=False,
)


c1_path, c1_raw = discover_table(
    required_alias_groups=[
        REVIEW_ALIASES,
        ASPECT_ALIASES,
        SENTIMENT_ALIASES,
    ],
    preferred_terms=(
        "c1",
        "mock",
        "synthetic",
        "integration",
    ),
    exclude_terms=(
        "selector_refinement",
        "selected_reviews",
        "pp1_evidence",
    ),
    prefer_largest=True,
)


try:
    phase_path, phase_raw = discover_table(
        required_alias_groups=[
            REVIEW_ALIASES,
            PRODUCT_ALIASES,
            PHASE_ALIASES,
        ],
        preferred_terms=(
            "selector",
            "selected",
            "audit",
            "proposed",
        ),
        exclude_terms=(
            "selector_refinement",
            "pp1_evidence",
        ),
        prefer_largest=False,
    )

except FileNotFoundError:
    phase_path = None
    phase_raw = None


print(
    "Selector-ready source:",
    relative_path(selector_path),
    selector_raw.shape,
)

print(
    "Baseline-selection source:",
    relative_path(baseline_path),
    baseline_raw.shape,
)

print(
    "C1 evidence source:",
    relative_path(c1_path),
    c1_raw.shape,
)

print(
    "Selection-phase audit:",
    relative_path(phase_path)
    if phase_path is not None
    else "NOT FOUND",
)


Selector-ready source: results\selector\selector_ready_reviews.csv (896, 12)
Baseline-selection source: results\selector_evaluation\baseline_selected_reviews.csv (300, 16)
C1 evidence source: results\integration\c1_mock_aspect_sentiment.csv (1792, 6)
Selection-phase audit: results\selector\selected_evidence_prototype.csv


## 4. Normalize schemas and recover Full Proposed baseline


In [23]:
selector = selector_raw.copy()
baseline_all = baseline_raw.copy()
c1 = c1_raw.copy()


selector_product_col = resolve_column(
    selector.columns,
    PRODUCT_ALIASES,
)

selector_review_col = resolve_column(
    selector.columns,
    REVIEW_ALIASES,
)

usefulness_col = resolve_column(
    selector.columns,
    USEFULNESS_ALIASES,
)

representativeness_col = resolve_column(
    selector.columns,
    REPRESENTATIVENESS_ALIASES,
)

selector_minority_col = resolve_column(
    selector.columns,
    MINORITY_GROUP_ALIASES,
    required=False,
)


baseline_product_col = resolve_column(
    baseline_all.columns,
    PRODUCT_ALIASES,
)

baseline_review_col = resolve_column(
    baseline_all.columns,
    REVIEW_ALIASES,
)

baseline_method_col = resolve_column(
    baseline_all.columns,
    METHOD_ALIASES,
)

baseline_rank_col = resolve_column(
    baseline_all.columns,
    RANK_ALIASES,
    required=False,
)


c1_review_col = resolve_column(
    c1.columns,
    REVIEW_ALIASES,
)

c1_aspect_col = resolve_column(
    c1.columns,
    ASPECT_ALIASES,
)

c1_sentiment_col = resolve_column(
    c1.columns,
    SENTIMENT_ALIASES,
)

c1_product_col = resolve_column(
    c1.columns,
    PRODUCT_ALIASES,
    required=False,
)


selector["_product"] = (
    selector[
        selector_product_col
    ].astype(str)
)

selector["_review"] = (
    selector[
        selector_review_col
    ].astype(str)
)


baseline_all["_product"] = (
    baseline_all[
        baseline_product_col
    ].astype(str)
)

baseline_all["_review"] = (
    baseline_all[
        baseline_review_col
    ].astype(str)
)


c1["_review"] = (
    c1[
        c1_review_col
    ].astype(str)
)


if c1_product_col is not None:
    c1["_product"] = (
        c1[
            c1_product_col
        ].astype(str)
    )

else:
    review_product_map = (
        selector[
            [
                "_review",
                "_product",
            ]
        ]
        .drop_duplicates(
            "_review"
        )
    )

    c1 = c1.merge(
        review_product_map,
        on="_review",
        how="left",
        validate="many_to_one",
    )


normalized_method = (
    baseline_all[
        baseline_method_col
    ]
    .astype(str)
    .map(normalize_name)
)


full_mask = (
    normalized_method.str.contains(
        "fullproposed",
        na=False,
    )
)


if not full_mask.any():
    full_mask = (
        normalized_method.str.contains(
            "full",
            na=False,
        )
        &
        normalized_method.str.contains(
            "proposed",
            na=False,
        )
    )


if not full_mask.any():
    raise ValueError(
        "Could not identify Full Proposed "
        "selection in baseline table."
    )


current_selected = (
    baseline_all.loc[
        full_mask
    ]
    .copy()
)


if baseline_rank_col is not None:
    current_selected = (
        current_selected.sort_values(
            [
                "_product",
                baseline_rank_col,
            ]
        )
    )

else:
    current_selected = (
        current_selected.sort_values(
            [
                "_product",
                "_review",
            ]
        )
    )


per_product_counts = (
    current_selected
    .groupby("_product")
    .size()
)


K = int(
    per_product_counts
    .mode()
    .iloc[0]
)


print(
    "Full Proposed rows:",
    len(current_selected),
)

print(
    "Products:",
    current_selected[
        "_product"
    ].nunique(),
)

print(
    "Detected budget k:",
    K,
)


display(
    per_product_counts
    .rename("selected_count")
    .reset_index()
)


Full Proposed rows: 50
Products: 10
Detected budget k: 5


,_product,selected_count
0,B07GYY6VTD,5
1,B07SDQ483V,5
2,B08PP9FM9N,5
3,B08XRDVWQ2,5
4,B09HFW8GRB,5
5,B09JJCXC2M,5
6,B09PPWVQSB,5
7,B0B8PDLMMS,5
8,B0BL9TKSBL,5
9,B0BMWTL2S6,5


In [24]:
def minmax_by_product(
    frame,
    value_col,
):
    numeric = pd.to_numeric(
        frame[value_col],
        errors="coerce",
    )


    def scale(series):
        minimum = series.min()
        maximum = series.max()

        if pd.isna(minimum):
            return pd.Series(
                np.nan,
                index=series.index,
            )

        if maximum == minimum:
            return pd.Series(
                0.5,
                index=series.index,
            )

        return (
            (series - minimum)
            /
            (maximum - minimum)
        )


    return (
        numeric
        .groupby(
            frame["_product"]
        )
        .transform(scale)
    )


selector[
    "_usefulness_norm"
] = minmax_by_product(
    selector,
    usefulness_col,
)


selector[
    "_representativeness_norm"
] = minmax_by_product(
    selector,
    representativeness_col,
)


selector[
    "_repair_utility"
] = (
    0.55
    * selector[
        "_usefulness_norm"
    ]
    +
    0.45
    * selector[
        "_representativeness_norm"
    ]
)


## 5. C1 contract diagnostics


In [25]:
c1["_aspect"] = (
    c1[
        c1_aspect_col
    ]
    .astype(str)
    .str.strip()
)

c1["_sentiment"] = (
    c1[
        c1_sentiment_col
    ]
    .astype(str)
    .str.strip()
)


duplicate_exact_rows = int(
    c1.duplicated(
        subset=[
            "_product",
            "_review",
            "_aspect",
            "_sentiment",
        ],
        keep=False,
    ).sum()
)


selector_review_ids = set(
    selector[
        "_review"
    ]
)

c1_review_ids = set(
    c1[
        "_review"
    ]
)


missing_c1_review_ids = (
    selector_review_ids
    - c1_review_ids
)


allowed_sentiments = {
    "positive",
    "neutral",
    "negative",
}


observed_sentiments = {
    str(value)
    .strip()
    .lower()

    for value
    in c1[
        "_sentiment"
    ].dropna()
}


unknown_sentiments = (
    observed_sentiments
    - allowed_sentiments
)


synthetic_flag_col = None

for candidate in [
    "is_mock",
    "is_synthetic",
    "synthetic",
]:
    if candidate in c1.columns:
        synthetic_flag_col = candidate
        break


if synthetic_flag_col is not None:

    c1_is_synthetic = bool(
        c1[
            synthetic_flag_col
        ]
        .fillna(False)
        .astype(bool)
        .any()
    )

else:
    c1_is_synthetic = any(
        term in str(
            c1_path
        ).lower()

        for term in [
            "mock",
            "synthetic",
            "fixture",
        ]
    )


c1_evidence = (
    c1[
        [
            "_product",
            "_review",
            "_aspect",
            "_sentiment",
        ]
    ]
    .dropna()
    .drop_duplicates()
    .copy()
)


c1_evidence[
    "_group"
] = (
    c1_evidence[
        "_aspect"
    ].astype(str)
    +
    "::"
    +
    c1_evidence[
        "_sentiment"
    ].astype(str)
)


contract_summary = pd.DataFrame(
    [
        {
            "check": "C1 rows",
            "value": len(c1),
        },
        {
            "check": "Unique C1 review IDs",
            "value": c1[
                "_review"
            ].nunique(),
        },
        {
            "check": "Exact duplicate rows",
            "value": duplicate_exact_rows,
        },
        {
            "check": "Selector IDs missing from C1",
            "value": len(
                missing_c1_review_ids
            ),
        },
        {
            "check": "Unknown sentiment labels",
            "value": len(
                unknown_sentiments
            ),
        },
        {
            "check": "Synthetic/mock C1",
            "value": c1_is_synthetic,
        },
    ]
)


display(contract_summary)


if c1_is_synthetic:
    print(
        "IMPORTANT: C1 evidence is synthetic/mock. "
        "C1-derived metrics are prototype diagnostics only."
    )


,check,value
0,C1 rows,1792
1,Unique C1 review IDs,896
2,Exact duplicate rows,0
3,Selector IDs missing from C1,0
4,Unknown sentiment labels,0
5,Synthetic/mock C1,True


IMPORTANT: C1 evidence is synthetic/mock. C1-derived metrics are prototype diagnostics only.


## 6. Recover qualified supported-minority groups

The notebook first attempts to load the existing group-level minority qualification
output produced earlier in the C2 pipeline.

If a suitable group-level qualification table is available, it is treated as the
authoritative prototype definition of qualified supported-minority groups.

A fallback is retained only for compatibility and is clearly reported.


In [26]:
minority_diag_path = None
minority_diag_raw = None
minority_supported_flag_col = None


minority_candidates = []


for path in all_result_tables():

    path_text = str(
        path
    ).lower()

    if "selector_refinement" in path_text:
        continue

    try:
        frame = read_table(path)
    except Exception:
        continue

    try:
        product_col = resolve_column(
            frame.columns,
            PRODUCT_ALIASES,
        )

        aspect_col = resolve_column(
            frame.columns,
            ASPECT_ALIASES,
        )

        sentiment_col = resolve_column(
            frame.columns,
            SENTIMENT_ALIASES,
        )

    except KeyError:
        continue


    supported_col = resolve_column(
        frame.columns,
        SUPPORTED_FLAG_ALIASES,
        required=False,
    )


    if supported_col is None:
        continue


    score = 0

    for term in [
        "minority",
        "group",
        "supported",
        "qualification",
        "integration",
    ]:
        if term in path_text:
            score += 1


    minority_candidates.append(
        (
            score,
            len(frame),
            path,
            frame,
            supported_col,
            product_col,
            aspect_col,
            sentiment_col,
        )
    )


if minority_candidates:

    minority_candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
        ),
        reverse=True,
    )


    (
        _,
        _,
        minority_diag_path,
        minority_diag_raw,
        minority_supported_flag_col,
        minority_product_col,
        minority_aspect_col,
        minority_sentiment_col,
    ) = minority_candidates[0]


    minority_diag = (
        minority_diag_raw.copy()
    )


    minority_diag[
        "_product"
    ] = (
        minority_diag[
            minority_product_col
        ].astype(str)
    )


    minority_diag[
        "_group"
    ] = (
        minority_diag[
            minority_aspect_col
        ].astype(str)
        +
        "::"
        +
        minority_diag[
            minority_sentiment_col
        ].astype(str)
    )


    minority_diag[
        "_is_supported"
    ] = (
        minority_diag[
            minority_supported_flag_col
        ]
        .apply(parse_bool)
    )


    supported_diag = (
        minority_diag[
            minority_diag[
                "_is_supported"
            ]
        ]
        .copy()
    )


    supported_groups_by_product = {
        product: set(
            frame[
                "_group"
            ]
        )

        for product, frame
        in supported_diag.groupby(
            "_product"
        )
    }


    supported_group_source = (
        "group_level_qualification_table"
    )


else:

    minority_diag = None
    supported_diag = None


    if selector_minority_col is None:

        supported_groups_by_product = {}

    else:

        selector[
            "_fallback_minority_groups"
        ] = (
            selector[
                selector_minority_col
            ]
            .apply(
                parse_group_collection
            )
        )


        supported_groups_by_product = {}


        for (
            product,
            frame,
        ) in selector.groupby(
            "_product"
        ):

            groups = set()

            for values in frame[
                "_fallback_minority_groups"
            ]:
                groups |= set(values)

            supported_groups_by_product[
                product
            ] = groups


    supported_group_source = (
        "selector_ready_fallback"
    )


supported_group_summary = pd.DataFrame(
    [
        {
            "product": product,
            "supported_minority_group_count": len(
                supported_groups_by_product.get(
                    product,
                    set(),
                )
            ),
        }

        for product in sorted(
            selector[
                "_product"
            ].unique()
        )
    ]
)


print(
    "Supported-minority source:",
    supported_group_source,
)

print(
    "Group diagnostic table:",
    relative_path(
        minority_diag_path
    )
    if minority_diag_path is not None
    else "Fallback used",
)


display(
    supported_group_summary
)


Supported-minority source: group_level_qualification_table
Group diagnostic table: results\integration\mock_supported_minority_groups.csv


,product,supported_minority_group_count
0,B07GYY6VTD,11
1,B07SDQ483V,11
2,B08PP9FM9N,9
3,B08XRDVWQ2,12
4,B09HFW8GRB,12
5,B09JJCXC2M,12
6,B09PPWVQSB,13
7,B0B8PDLMMS,9
8,B0BL9TKSBL,12
9,B0BMWTL2S6,10


In [27]:
review_supported_groups = {}


for (
    product,
    review_id,
), frame in c1_evidence.groupby(
    [
        "_product",
        "_review",
    ]
):

    supported_groups = (
        supported_groups_by_product.get(
            product,
            set(),
        )
    )

    review_groups = set(
        frame[
            "_group"
        ]
    )

    review_supported_groups[
        (
            product,
            review_id,
        )
    ] = (
        review_groups
        &
        supported_groups
    )


def groups_for_reviews(
    product,
    review_ids,
):
    result = set()

    for review_id in review_ids:

        result |= (
            review_supported_groups.get(
                (
                    str(product),
                    str(review_id),
                ),
                set(),
            )
        )

    return result


## 7. Recover and lock minority-reserve reviews

Fidelity repair is applied **after** the current Full Proposed selector.

Reviews that were selected in the existing minority-reserve phase are therefore
recovered and locked. Fidelity repair is allowed to replace only general-selection
reviews.

This preserves the current selector's explicit minority-preservation action without
attempting to infer a different minority policy.


In [28]:
baseline_ids_by_product = {
    str(product): (
        frame[
            "_review"
        ]
        .astype(str)
        .tolist()
    )

    for product, frame
    in current_selected.groupby(
        "_product",
        sort=True,
    )
}


locked_minority_reviews_by_product = {
    product: set()
    for product
    in baseline_ids_by_product
}


reserve_recovery_source = (
    "not_recovered"
)


if phase_raw is not None:

    phase_df = (
        phase_raw.copy()
    )


    phase_product_col = resolve_column(
        phase_df.columns,
        PRODUCT_ALIASES,
    )

    phase_review_col = resolve_column(
        phase_df.columns,
        REVIEW_ALIASES,
    )

    phase_col = resolve_column(
        phase_df.columns,
        PHASE_ALIASES,
    )

    phase_method_col = resolve_column(
        phase_df.columns,
        METHOD_ALIASES,
        required=False,
    )


    phase_df[
        "_product"
    ] = (
        phase_df[
            phase_product_col
        ].astype(str)
    )

    phase_df[
        "_review"
    ] = (
        phase_df[
            phase_review_col
        ].astype(str)
    )


    if phase_method_col is not None:

        phase_method_norm = (
            phase_df[
                phase_method_col
            ]
            .astype(str)
            .map(normalize_name)
        )


        phase_full_mask = (
            phase_method_norm
            .str.contains(
                "fullproposed",
                na=False,
            )
        )


        if phase_full_mask.any():

            phase_df = (
                phase_df.loc[
                    phase_full_mask
                ]
                .copy()
            )


    reserve_mask = (
        phase_df[
            phase_col
        ]
        .astype(str)
        .str.lower()
        .str.contains(
            r"minority|reserve",
            regex=True,
            na=False,
        )
    )


    reserve_rows = (
        phase_df.loc[
            reserve_mask
        ]
        .copy()
    )


    current_pairs = set(
        zip(
            current_selected[
                "_product"
            ],
            current_selected[
                "_review"
            ],
        )
    )


    reserve_rows[
        "_is_current_baseline_pair"
    ] = [
        (
            product,
            review_id,
        )
        in current_pairs

        for product, review_id
        in zip(
            reserve_rows[
                "_product"
            ],
            reserve_rows[
                "_review"
            ],
        )
    ]


    reserve_rows = (
        reserve_rows[
            reserve_rows[
                "_is_current_baseline_pair"
            ]
        ]
        .copy()
    )


    for (
        product,
        frame,
    ) in reserve_rows.groupby(
        "_product"
    ):

        locked_minority_reviews_by_product[
            str(product)
        ] = set(
            frame[
                "_review"
            ].astype(str)
        )


    reserve_recovery_source = (
        "selection_phase_audit"
    )


reserve_summary_rows = []


for product in sorted(
    baseline_ids_by_product
):

    supported_groups = (
        supported_groups_by_product.get(
            product,
            set(),
        )
    )

    locked_reviews = (
        locked_minority_reviews_by_product.get(
            product,
            set(),
        )
    )

    protected_groups = (
        groups_for_reviews(
            product,
            locked_reviews,
        )
    )


    reserve_summary_rows.append(
        {
            "product": product,
            "supported_group_count": len(
                supported_groups
            ),
            "locked_minority_review_count": len(
                locked_reviews
            ),
            "locked_minority_review_ids": sorted(
                locked_reviews
            ),
            "protected_group_count": len(
                protected_groups
            ),
            "protected_groups": sorted(
                protected_groups
            ),
            "recovery_source": (
                reserve_recovery_source
            ),
        }
    )


reserve_summary = pd.DataFrame(
    reserve_summary_rows
)


display(
    reserve_summary
)


products_with_supported_groups = (
    reserve_summary[
        reserve_summary[
            "supported_group_count"
        ] > 0
    ]
)


unrecovered_products = (
    products_with_supported_groups[
        products_with_supported_groups[
            "locked_minority_review_count"
        ] == 0
    ][
        "product"
    ].tolist()
)


print(
    "Products with supported minority groups:",
    len(
        products_with_supported_groups
    ),
)

print(
    "Products where minority reserve was not recovered:",
    unrecovered_products,
)


,product,supported_group_count,locked_minority_review_count,locked_minority_review_ids,protected_group_count,protected_groups,recovery_source
0,B07GYY6VTD,11,2,"[082b67f5c74a45b8, 866978504b0f427f]",3,"[camera::negative, display::neutral, performan...",selection_phase_audit
1,B07SDQ483V,11,2,"[03fe2e44ba4656ed, 8ea0f6f697d7447e]",3,"[battery::neutral, camera::negative, design::n...",selection_phase_audit
2,B08PP9FM9N,9,2,"[1e56e51357613656, b5bd4bd905712bd4]",3,"[battery::negative, camera::negative, camera::...",selection_phase_audit
3,B08XRDVWQ2,12,2,"[2b39bf6748f95a3f, da37980592c022c6]",4,"[battery::negative, battery::neutral, camera::...",selection_phase_audit
4,B09HFW8GRB,12,2,"[791cfcb377c79644, 7c4cc5fa963e85a2]",3,"[battery::negative, camera::negative, performa...",selection_phase_audit
5,B09JJCXC2M,12,2,"[01c2860c638f70e1, 241d5c5722c38a8c]",2,"[battery::neutral, performance::neutral]",selection_phase_audit
6,B09PPWVQSB,13,2,"[640ed5cdcfce8432, 80815f33689b51a0]",3,"[design::neutral, display::negative, price::ne...",selection_phase_audit
7,B0B8PDLMMS,9,2,"[31969113ff51dda3, a8db021ec28ce162]",2,"[camera::negative, performance::neutral]",selection_phase_audit
8,B0BL9TKSBL,12,2,"[7749273cf3a0cc69, b31ee5f2f615eb79]",3,"[camera::neutral, display::neutral, performanc...",selection_phase_audit
9,B0BMWTL2S6,10,2,"[00e860b237548ac0, 7ed060fd651a1e33]",3,"[camera::negative, display::negative, performa...",selection_phase_audit


Products with supported minority groups: 10
Products where minority reserve was not recovered: []


## 8. Load semantic embeddings for redundancy diagnostics


In [29]:
embedding_path = None
embedding_index_path = None
embeddings = None
embedding_lookup = None


npy_candidates = sorted(
    list(
        RESULTS_ROOT.rglob(
            "*.npy"
        )
    ),
    key=lambda path: (
        "embedding"
        in str(path).lower(),
        -len(str(path)),
    ),
    reverse=True,
)


if npy_candidates:

    embedding_path = (
        npy_candidates[0]
    )

    embeddings = np.load(
        embedding_path,
        allow_pickle=False,
    )


    index_candidates = []


    for path in RESULTS_ROOT.rglob(
        "*.csv"
    ):

        path_text = str(
            path
        ).lower()

        if (
            "embedding"
            not in path_text
            or
            "index"
            not in path_text
        ):
            continue


        try:
            frame = pd.read_csv(
                path
            )

            review_col = resolve_column(
                frame.columns,
                REVIEW_ALIASES,
            )

        except Exception:
            continue


        index_candidates.append(
            (
                abs(
                    len(frame)
                    -
                    len(embeddings)
                ),
                path,
                frame,
                review_col,
            )
        )


    if index_candidates:

        index_candidates.sort(
            key=lambda item: (
                item[0],
                len(str(item[1])),
            )
        )


        (
            _,
            embedding_index_path,
            embedding_index,
            embedding_review_col,
        ) = index_candidates[0]


        if (
            len(embedding_index)
            ==
            len(embeddings)
        ):

            embedding_lookup = {
                str(review_id): index

                for index, review_id
                in enumerate(
                    embedding_index[
                        embedding_review_col
                    ].astype(str)
                )
            }


print(
    "Embedding source:",
    relative_path(
        embedding_path
    ),
)

print(
    "Embedding index:",
    relative_path(
        embedding_index_path
    ),
)

print(
    "Embedding diagnostics available:",
    embedding_lookup
    is not None,
)


Embedding source: results\integration\smartphone_review_embeddings.npy
Embedding index: results\integration\smartphone_review_embedding_index.csv
Embedding diagnostics available: True


## 9. Evaluation metrics

Opinion fidelity is measured using Jensen-Shannon Divergence between the full
product-level aspect-sentiment distribution and the selected-set distribution.

Lower JSD indicates better opinion fidelity.


In [30]:
def opinion_jsd(
    product,
    review_ids,
):
    product = str(product)

    review_ids = {
        str(value)
        for value
        in review_ids
    }


    base = (
        c1_evidence[
            c1_evidence[
                "_product"
            ] == product
        ]
    )


    selected = (
        base[
            base[
                "_review"
            ].isin(
                review_ids
            )
        ]
    )


    if (
        base.empty
        or
        selected.empty
    ):
        return np.nan


    keys = sorted(
        set(
            base[
                "_group"
            ].unique()
        )
    )


    p = (
        base[
            "_group"
        ]
        .value_counts()
        .reindex(
            keys,
            fill_value=0,
        )
        .astype(float)
    )


    q = (
        selected[
            "_group"
        ]
        .value_counts()
        .reindex(
            keys,
            fill_value=0,
        )
        .astype(float)
    )


    p = (
        p
        /
        p.sum()
    ).to_numpy()


    q = (
        q
        /
        q.sum()
    ).to_numpy()


    return float(
        jensenshannon(
            p,
            q,
            base=2.0,
        ) ** 2
    )


def aspect_coverage(
    product,
    review_ids,
):
    product = str(product)

    review_ids = {
        str(value)
        for value
        in review_ids
    }


    base = (
        c1_evidence[
            c1_evidence[
                "_product"
            ] == product
        ]
    )


    all_aspects = set(
        base[
            "_aspect"
        ]
    )


    if not all_aspects:
        return np.nan


    selected_aspects = set(
        base[
            base[
                "_review"
            ].isin(
                review_ids
            )
        ][
            "_aspect"
        ]
    )


    return (
        len(
            selected_aspects
        )
        /
        len(
            all_aspects
        )
    )


def supported_minority_recall(
    product,
    review_ids,
):
    product = str(product)

    available = (
        supported_groups_by_product.get(
            product,
            set(),
        )
    )


    if not available:
        return np.nan


    covered = groups_for_reviews(
        product,
        review_ids,
    )


    return (
        len(
            available
            &
            covered
        )
        /
        len(
            available
        )
    )


def protected_minority_coverage(
    product,
    review_ids,
):
    product = str(product)

    locked = (
        locked_minority_reviews_by_product.get(
            product,
            set(),
        )
    )


    if not locked:
        return np.nan


    required_groups = groups_for_reviews(
        product,
        locked,
    )


    if not required_groups:
        return np.nan


    covered = groups_for_reviews(
        product,
        review_ids,
    )


    return (
        len(
            required_groups
            &
            covered
        )
        /
        len(
            required_groups
        )
    )


def locked_review_retention(
    product,
    review_ids,
):
    product = str(product)

    locked = (
        locked_minority_reviews_by_product.get(
            product,
            set(),
        )
    )


    if not locked:
        return np.nan


    selected = {
        str(value)
        for value
        in review_ids
    }


    return (
        len(
            locked
            &
            selected
        )
        /
        len(
            locked
        )
    )


def mean_pairwise_similarity(
    review_ids,
):

    if embedding_lookup is None:
        return np.nan


    indices = [
        embedding_lookup[
            str(review_id)
        ]

        for review_id
        in review_ids

        if str(review_id)
        in embedding_lookup
    ]


    if len(indices) < 2:
        return 0.0


    matrix = embeddings[
        indices
    ]


    similarity = cosine_similarity(
        matrix
    )


    upper = similarity[
        np.triu_indices(
            len(indices),
            k=1,
        )
    ]


    return float(
        upper.mean()
    )


def evaluation_row(
    product,
    method,
    review_ids,
):
    product = str(product)

    review_ids = [
        str(value)
        for value
        in review_ids
    ]


    rows = selector[
        (
            selector[
                "_product"
            ]
            ==
            product
        )
        &
        (
            selector[
                "_review"
            ].isin(
                review_ids
            )
        )
    ]


    return {
        "product": product,
        "method": method,
        "selected_count": len(
            review_ids
        ),
        "mean_usefulness": float(
            rows[
                "_usefulness_norm"
            ].mean()
        ),
        "mean_representativeness": float(
            rows[
                "_representativeness_norm"
            ].mean()
        ),
        "aspect_coverage": aspect_coverage(
            product,
            review_ids,
        ),
        "supported_minority_recall": supported_minority_recall(
            product,
            review_ids,
        ),
        "protected_minority_coverage": protected_minority_coverage(
            product,
            review_ids,
        ),
        "locked_review_retention": locked_review_retention(
            product,
            review_ids,
        ),
        "mean_pairwise_cosine_similarity": mean_pairwise_similarity(
            review_ids
        ),
        "opinion_jsd": opinion_jsd(
            product,
            review_ids,
        ),
    }


## 10. Prototype fidelity configuration

The default JSD threshold of `0.35` remains a provisional PP1 configuration.
It is not a final research threshold.


In [31]:
FIDELITY_THRESHOLD = 0.35

FIDELITY_SENSITIVITY_THRESHOLDS = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
]

MAX_REPAIR_SWAPS = 3

MAX_MEAN_UTILITY_DROP = 0.10


print(
    "Budget k:",
    K,
)

print(
    "Default fidelity threshold:",
    FIDELITY_THRESHOLD,
)

print(
    "Maximum repair swaps:",
    MAX_REPAIR_SWAPS,
)


Budget k: 5
Default fidelity threshold: 0.35
Maximum repair swaps: 3


## 11. Constraint pre-check


In [32]:
def constraint_precheck(
    product,
    candidate_ids,
    baseline_ids,
    k,
):
    product = str(product)

    candidate_ids = {
        str(value)
        for value
        in candidate_ids
    }

    baseline_ids = {
        str(value)
        for value
        in baseline_ids
    }


    if k <= 0:
        return {
            "status": "failed",
            "reason": "invalid_budget",
        }


    if len(candidate_ids) < k:
        return {
            "status": "failed",
            "reason": "insufficient_candidates",
        }


    if len(baseline_ids) != k:
        return {
            "status": "failed",
            "reason": "baseline_selection_count_mismatch",
        }


    if not baseline_ids.issubset(
        candidate_ids
    ):
        return {
            "status": "failed",
            "reason": "baseline_review_missing_from_candidate_pool",
        }


    supported_groups = (
        supported_groups_by_product.get(
            product,
            set(),
        )
    )


    locked_reviews = (
        locked_minority_reviews_by_product.get(
            product,
            set(),
        )
    )


    if not supported_groups:

        return {
            "status": "no_minority_required",
            "reason": "no_qualified_supported_minority_groups",
        }


    if not locked_reviews:

        return {
            "status": "failed",
            "reason": "supported_minority_exists_but_reserve_not_recovered",
        }


    if not locked_reviews.issubset(
        baseline_ids
    ):

        return {
            "status": "failed",
            "reason": "minority_reserve_review_not_in_baseline",
        }


    if len(
        locked_reviews
    ) > k:

        return {
            "status": "failed",
            "reason": "minority_reserve_exceeds_budget",
        }


    return {
        "status": "feasible",
        "reason": "precheck_passed",
    }


## 12. Fidelity-aware bounded repair

Repair policy:

1. Start from the existing Full Proposed set.
2. Recover minority-reserve reviews.
3. Lock those minority-reserve reviews.
4. Check final-set JSD.
5. If fidelity is above the provisional threshold, search one-for-one swaps only among
   non-reserved selected reviews.
6. Accept only deterministic JSD-improving swaps within the configured utility-loss bound.
7. Stop when fidelity is satisfied, no valid improving swap exists, or the swap limit is reached.
8. Return explicit `feasible`, `relaxed`, `failed`, or `no_minority_required` status.


In [33]:
def mean_repair_utility(
    product,
    review_ids,
):
    rows = selector[
        (
            selector[
                "_product"
            ]
            ==
            str(product)
        )
        &
        (
            selector[
                "_review"
            ].isin(
                [
                    str(value)
                    for value
                    in review_ids
                ]
            )
        )
    ]


    return float(
        rows[
            "_repair_utility"
        ].mean()
    )


def refine_product_selection(
    product,
    baseline_review_ids,
    fidelity_threshold=FIDELITY_THRESHOLD,
    max_swaps=MAX_REPAIR_SWAPS,
    max_mean_utility_drop=MAX_MEAN_UTILITY_DROP,
):
    product = str(product)


    candidate_frame = (
        selector[
            selector[
                "_product"
            ]
            ==
            product
        ]
        .drop_duplicates(
            "_review"
        )
        .copy()
    )


    candidate_ids = (
        candidate_frame[
            "_review"
        ]
        .astype(str)
        .tolist()
    )


    baseline_review_ids = [
        str(value)
        for value
        in baseline_review_ids
    ]


    precheck = constraint_precheck(
        product=product,
        candidate_ids=candidate_ids,
        baseline_ids=baseline_review_ids,
        k=K,
    )


    initial_jsd = opinion_jsd(
        product,
        baseline_review_ids,
    )


    if precheck[
        "status"
    ] == "failed":

        return {
            "product": product,
            "status": "failed",
            "detail": precheck[
                "reason"
            ],
            "relaxed_condition": None,
            "selected_ids": baseline_review_ids,
            "initial_jsd": initial_jsd,
            "final_jsd": initial_jsd,
            "repair_swaps": 0,
            "repair_trace": [],
        }


    locked_reviews = set(
        locked_minority_reviews_by_product.get(
            product,
            set(),
        )
    )


    selected_ids = list(
        baseline_review_ids
    )


    base_utility = mean_repair_utility(
        product,
        selected_ids,
    )


    if pd.isna(
        initial_jsd
    ):

        return {
            "product": product,
            "status": "failed",
            "detail": "opinion_fidelity_not_computable",
            "relaxed_condition": None,
            "selected_ids": selected_ids,
            "initial_jsd": initial_jsd,
            "final_jsd": initial_jsd,
            "repair_swaps": 0,
            "repair_trace": [],
        }


    current_jsd = (
        initial_jsd
    )

    repair_trace = []


    for swap_number in range(
        1,
        max_swaps + 1,
    ):

        if (
            current_jsd
            <=
            fidelity_threshold
        ):
            break


        selected_set = set(
            selected_ids
        )


        removable_positions = [
            index

            for index, review_id
            in enumerate(
                selected_ids
            )

            if review_id
            not in locked_reviews
        ]


        unselected_ids = sorted(
            set(
                candidate_ids
            )
            -
            selected_set
        )


        best = None


        for remove_position in (
            removable_positions
        ):

            remove_id = (
                selected_ids[
                    remove_position
                ]
            )


            for add_id in (
                unselected_ids
            ):

                trial_ids = list(
                    selected_ids
                )

                trial_ids[
                    remove_position
                ] = add_id


                trial_set = set(
                    trial_ids
                )


                # Hard protection:
                # minority-reserve reviews cannot disappear.
                if not locked_reviews.issubset(
                    trial_set
                ):
                    continue


                trial_jsd = opinion_jsd(
                    product,
                    trial_ids,
                )


                if pd.isna(
                    trial_jsd
                ):
                    continue


                if (
                    trial_jsd
                    >=
                    current_jsd
                    - 1e-12
                ):
                    continue


                trial_utility = (
                    mean_repair_utility(
                        product,
                        trial_ids,
                    )
                )


                utility_drop = max(
                    0.0,
                    float(
                        base_utility
                        -
                        trial_utility
                    ),
                )


                if (
                    utility_drop
                    >
                    max_mean_utility_drop
                ):
                    continue


                minority_recall = (
                    supported_minority_recall(
                        product,
                        trial_ids,
                    )
                )


                protected_coverage = (
                    protected_minority_coverage(
                        product,
                        trial_ids,
                    )
                )


                retention = (
                    locked_review_retention(
                        product,
                        trial_ids,
                    )
                )


                # Lowest JSD first.
                # Then prefer smaller utility loss.
                # Then prefer better minority recall.
                key = (
                    float(
                        trial_jsd
                    ),
                    float(
                        utility_drop
                    ),
                    (
                        -float(
                            minority_recall
                        )
                        if not pd.isna(
                            minority_recall
                        )
                        else 0.0
                    ),
                    str(
                        add_id
                    ),
                    str(
                        remove_id
                    ),
                )


                if (
                    best is None
                    or
                    key < best[
                        "key"
                    ]
                ):

                    best = {
                        "key": key,
                        "remove_position": remove_position,
                        "remove_id": remove_id,
                        "add_id": add_id,
                        "trial_ids": trial_ids,
                        "trial_jsd": float(
                            trial_jsd
                        ),
                        "utility_drop": float(
                            utility_drop
                        ),
                        "minority_recall": (
                            float(
                                minority_recall
                            )
                            if not pd.isna(
                                minority_recall
                            )
                            else np.nan
                        ),
                        "protected_coverage": (
                            float(
                                protected_coverage
                            )
                            if not pd.isna(
                                protected_coverage
                            )
                            else np.nan
                        ),
                        "locked_retention": (
                            float(
                                retention
                            )
                            if not pd.isna(
                                retention
                            )
                            else np.nan
                        ),
                    }


        if best is None:
            break


        previous_jsd = (
            current_jsd
        )


        selected_ids = (
            best[
                "trial_ids"
            ]
        )


        current_jsd = (
            best[
                "trial_jsd"
            ]
        )


        repair_trace.append(
            {
                "product": product,
                "swap_number": swap_number,
                "removed_review_id": best[
                    "remove_id"
                ],
                "added_review_id": best[
                    "add_id"
                ],
                "jsd_before": previous_jsd,
                "jsd_after": current_jsd,
                "utility_drop_from_initial": best[
                    "utility_drop"
                ],
                "supported_minority_recall_after": best[
                    "minority_recall"
                ],
                "protected_minority_coverage_after": best[
                    "protected_coverage"
                ],
                "locked_review_retention_after": best[
                    "locked_retention"
                ],
            }
        )


    final_jsd = opinion_jsd(
        product,
        selected_ids,
    )


    final_retention = (
        locked_review_retention(
            product,
            selected_ids,
        )
    )


    if (
        locked_reviews
        and
        final_retention
        != 1.0
    ):

        return {
            "product": product,
            "status": "failed",
            "detail": "minority_reserve_not_preserved",
            "relaxed_condition": None,
            "selected_ids": selected_ids,
            "initial_jsd": initial_jsd,
            "final_jsd": final_jsd,
            "repair_swaps": len(
                repair_trace
            ),
            "repair_trace": repair_trace,
        }


    if (
        final_jsd
        <=
        fidelity_threshold
    ):

        if (
            precheck[
                "status"
            ]
            ==
            "no_minority_required"
        ):

            status = (
                "no_minority_required"
            )

        else:

            status = (
                "feasible"
            )


        detail = (
            "fidelity_satisfied_after_repair"
            if repair_trace
            else
            "fidelity_satisfied_without_repair"
        )

        relaxed_condition = None


    else:

        status = (
            "relaxed"
        )

        detail = (
            "minority_reserve_preserved_but_fidelity_threshold_not_met"
            if locked_reviews
            else
            "fidelity_threshold_not_met"
        )

        relaxed_condition = (
            "opinion_fidelity"
        )


    return {
        "product": product,
        "status": status,
        "detail": detail,
        "relaxed_condition": relaxed_condition,
        "selected_ids": selected_ids,
        "initial_jsd": initial_jsd,
        "final_jsd": final_jsd,
        "repair_swaps": len(
            repair_trace
        ),
        "repair_trace": repair_trace,
    }


## 13. Run default fidelity-aware refinement


In [34]:
default_results = {}


for product in sorted(
    baseline_ids_by_product
):

    default_results[
        product
    ] = (
        refine_product_selection(
            product=product,
            baseline_review_ids=(
                baseline_ids_by_product[
                    product
                ]
            ),
        )
    )


outcome_rows = []
repair_rows = []


for (
    product,
    result,
) in default_results.items():

    locked_reviews = (
        locked_minority_reviews_by_product.get(
            product,
            set(),
        )
    )


    outcome_rows.append(
        {
            "product": product,
            "status": result[
                "status"
            ],
            "detail": result[
                "detail"
            ],
            "relaxed_condition": result[
                "relaxed_condition"
            ],
            "initial_jsd": result[
                "initial_jsd"
            ],
            "final_jsd": result[
                "final_jsd"
            ],
            "repair_swaps": result[
                "repair_swaps"
            ],
            "selected_count": len(
                result[
                    "selected_ids"
                ]
            ),
            "locked_minority_review_count": len(
                locked_reviews
            ),
            "locked_review_retention": (
                locked_review_retention(
                    product,
                    result[
                        "selected_ids"
                    ],
                )
            ),
            "protected_minority_coverage": (
                protected_minority_coverage(
                    product,
                    result[
                        "selected_ids"
                    ],
                )
            ),
        }
    )


    repair_rows.extend(
        result[
            "repair_trace"
        ]
    )


outcome_table = pd.DataFrame(
    outcome_rows
)

repair_trace_table = pd.DataFrame(
    repair_rows
)


display(
    outcome_table
)


print(
    "\nOutcome counts:"
)


display(
    outcome_table[
        "status"
    ]
    .value_counts()
    .rename_axis(
        "status"
    )
    .reset_index(
        name="count"
    )
)


,product,status,detail,relaxed_condition,initial_jsd,final_jsd,repair_swaps,selected_count,locked_minority_review_count,locked_review_retention,protected_minority_coverage
0,B07GYY6VTD,feasible,fidelity_satisfied_after_repair,None,0.387555,0.258250,1,5,2,1.0,1.0
1,B07SDQ483V,feasible,fidelity_satisfied_without_repair,None,0.269152,0.269152,0,5,2,1.0,1.0
2,B08PP9FM9N,feasible,fidelity_satisfied_after_repair,None,0.393369,0.232365,1,5,2,1.0,1.0
3,B08XRDVWQ2,feasible,fidelity_satisfied_after_repair,None,0.366642,0.245519,1,5,2,1.0,1.0
4,B09HFW8GRB,feasible,fidelity_satisfied_after_repair,None,0.376623,0.261142,1,5,2,1.0,1.0
5,B09JJCXC2M,feasible,fidelity_satisfied_after_repair,None,0.405713,0.269284,1,5,2,1.0,1.0
6,B09PPWVQSB,feasible,fidelity_satisfied_without_repair,None,0.289642,0.289642,0,5,2,1.0,1.0
7,B0B8PDLMMS,feasible,fidelity_satisfied_without_repair,None,0.284042,0.284042,0,5,2,1.0,1.0
8,B0BL9TKSBL,feasible,fidelity_satisfied_after_repair,None,0.369518,0.245825,1,5,2,1.0,1.0
9,B0BMWTL2S6,feasible,fidelity_satisfied_without_repair,None,0.225451,0.225451,0,5,2,1.0,1.0



Outcome counts:


,status,count
0,feasible,10


## 14. Build refined selected-review audit


In [35]:
refined_rows = []


for product in sorted(
    default_results
):

    result = (
        default_results[
            product
        ]
    )

    locked_reviews = (
        locked_minority_reviews_by_product.get(
            product,
            set(),
        )
    )


    for (
        rank,
        review_id,
    ) in enumerate(
        result[
            "selected_ids"
        ],
        start=1,
    ):

        match = selector[
            (
                selector[
                    "_product"
                ]
                ==
                product
            )
            &
            (
                selector[
                    "_review"
                ]
                ==
                review_id
            )
        ]


        if match.empty:
            continue


        row = (
            match.iloc[0]
            .to_dict()
        )


        row[
            "refined_selection_order"
        ] = rank

        row[
            "refinement_method"
        ] = (
            "FidelityAwareMinorityProtected"
        )

        row[
            "outcome_status"
        ] = result[
            "status"
        ]

        row[
            "fidelity_threshold"
        ] = (
            FIDELITY_THRESHOLD
        )

        row[
            "repair_swaps"
        ] = result[
            "repair_swaps"
        ]

        row[
            "baseline_selected"
        ] = (
            review_id
            in
            set(
                baseline_ids_by_product[
                    product
                ]
            )
        )

        row[
            "minority_reserve_locked"
        ] = (
            review_id
            in locked_reviews
        )


        refined_rows.append(
            row
        )


refined_selected = pd.DataFrame(
    refined_rows
)


print(
    "Refined selected rows:",
    len(
        refined_selected
    ),
)


display(
    refined_selected[
        [
            "_product",
            "_review",
            "refined_selection_order",
            "minority_reserve_locked",
            "baseline_selected",
            "outcome_status",
        ]
    ].head(20)
)


Refined selected rows: 50


,_product,_review,refined_selection_order,minority_reserve_locked,baseline_selected,outcome_status
0,B07GYY6VTD,082b67f5c74a45b8,1,True,True,feasible
1,B07GYY6VTD,271bfe42dc019eed,2,False,False,feasible
2,B07GYY6VTD,866978504b0f427f,3,True,True,feasible
3,B07GYY6VTD,b068d451b1f068c1,4,False,True,feasible
4,B07GYY6VTD,b8fc62a6d1a0d912,5,False,True,feasible
5,B07SDQ483V,03fe2e44ba4656ed,1,True,True,feasible
6,B07SDQ483V,092068c7d5262747,2,False,True,feasible
7,B07SDQ483V,72b8dd78d5aeed90,3,False,True,feasible
8,B07SDQ483V,745c41e302104325,4,False,True,feasible
9,B07SDQ483V,8ea0f6f697d7447e,5,True,True,feasible


## 15. Current Full Proposed vs Fidelity-Aware Minority-Protected selector


In [36]:
comparison_rows = []


for product in sorted(
    baseline_ids_by_product
):

    baseline_ids = (
        baseline_ids_by_product[
            product
        ]
    )

    refined_ids = (
        default_results[
            product
        ][
            "selected_ids"
        ]
    )


    baseline_metrics = (
        evaluation_row(
            product,
            "CurrentFullProposed",
            baseline_ids,
        )
    )

    baseline_metrics[
        "outcome_status"
    ] = "baseline"


    refined_metrics = (
        evaluation_row(
            product,
            "FidelityAwareMinorityProtected",
            refined_ids,
        )
    )

    refined_metrics[
        "outcome_status"
    ] = (
        default_results[
            product
        ][
            "status"
        ]
    )


    comparison_rows.extend(
        [
            baseline_metrics,
            refined_metrics,
        ]
    )


comparison_by_product = (
    pd.DataFrame(
        comparison_rows
    )
)


metric_columns = [
    "selected_count",
    "mean_usefulness",
    "mean_representativeness",
    "aspect_coverage",
    "supported_minority_recall",
    "protected_minority_coverage",
    "locked_review_retention",
    "mean_pairwise_cosine_similarity",
    "opinion_jsd",
]


overall_rows = []


for (
    method,
    frame,
) in comparison_by_product.groupby(
    "method"
):

    row = {
        "product": "__OVERALL__",
        "method": method,
        "outcome_status": "summary",
    }


    for column in metric_columns:

        row[
            column
        ] = float(
            pd.to_numeric(
                frame[
                    column
                ],
                errors="coerce",
            ).mean()
        )


    overall_rows.append(
        row
    )


comparison_overall = (
    pd.DataFrame(
        overall_rows
    )
)


fidelity_selector_comparison = (
    pd.concat(
        [
            comparison_by_product,
            comparison_overall,
        ],
        ignore_index=True,
    )
)


print(
    "Overall comparison:"
)

display(
    comparison_overall
)


Overall comparison:


,product,method,outcome_status,selected_count,mean_usefulness,mean_representativeness,aspect_coverage,supported_minority_recall,protected_minority_coverage,locked_review_retention,mean_pairwise_cosine_similarity,opinion_jsd
0,__OVERALL__,CurrentFullProposed,summary,5.0,0.742725,0.766597,0.933333,0.416744,1.0,1.0,0.492611,0.336771
1,__OVERALL__,FidelityAwareMinorityProtected,summary,5.0,0.711629,0.760175,0.950000,0.407653,1.0,1.0,0.480485,0.258067


## 16. Fidelity-threshold sensitivity

This is a prototype sensitivity analysis only.


In [37]:
sensitivity_rows = []


for threshold in (
    FIDELITY_SENSITIVITY_THRESHOLDS
):

    threshold_metrics = []


    for product in sorted(
        baseline_ids_by_product
    ):

        result = (
            refine_product_selection(
                product=product,
                baseline_review_ids=(
                    baseline_ids_by_product[
                        product
                    ]
                ),
                fidelity_threshold=threshold,
            )
        )


        metrics = evaluation_row(
            product,
            "FidelityAwareMinorityProtected",
            result[
                "selected_ids"
            ],
        )


        metrics[
            "status"
        ] = result[
            "status"
        ]

        metrics[
            "repair_swaps"
        ] = result[
            "repair_swaps"
        ]


        threshold_metrics.append(
            metrics
        )


    threshold_frame = pd.DataFrame(
        threshold_metrics
    )


    status_counts = (
        threshold_frame[
            "status"
        ].value_counts()
    )


    sensitivity_rows.append(
        {
            "fidelity_threshold": threshold,
            "mean_usefulness": float(
                threshold_frame[
                    "mean_usefulness"
                ].mean()
            ),
            "mean_representativeness": float(
                threshold_frame[
                    "mean_representativeness"
                ].mean()
            ),
            "mean_aspect_coverage": float(
                threshold_frame[
                    "aspect_coverage"
                ].mean()
            ),
            "mean_supported_minority_recall": float(
                threshold_frame[
                    "supported_minority_recall"
                ].mean()
            ),
            "mean_protected_minority_coverage": float(
                threshold_frame[
                    "protected_minority_coverage"
                ].mean()
            ),
            "mean_locked_review_retention": float(
                threshold_frame[
                    "locked_review_retention"
                ].mean()
            ),
            "mean_pairwise_cosine_similarity": float(
                threshold_frame[
                    "mean_pairwise_cosine_similarity"
                ].mean()
            ),
            "mean_opinion_jsd": float(
                threshold_frame[
                    "opinion_jsd"
                ].mean()
            ),
            "total_repair_swaps": int(
                threshold_frame[
                    "repair_swaps"
                ].sum()
            ),
            "feasible_products": int(
                status_counts.get(
                    "feasible",
                    0,
                )
            ),
            "no_minority_required_products": int(
                status_counts.get(
                    "no_minority_required",
                    0,
                )
            ),
            "relaxed_products": int(
                status_counts.get(
                    "relaxed",
                    0,
                )
            ),
            "failed_products": int(
                status_counts.get(
                    "failed",
                    0,
                )
            ),
        }
    )


fidelity_sensitivity = (
    pd.DataFrame(
        sensitivity_rows
    )
)


display(
    fidelity_sensitivity
)


,fidelity_threshold,mean_usefulness,mean_representativeness,mean_aspect_coverage,mean_supported_minority_recall,mean_protected_minority_coverage,mean_locked_review_retention,mean_pairwise_cosine_similarity,mean_opinion_jsd,total_repair_swaps,feasible_products,no_minority_required_products,relaxed_products,failed_products
0,0.20,0.627325,0.788767,1.000000,0.362300,1.0,1.0,0.504547,0.205620,18,5,0,5,0
1,0.25,0.662012,0.770956,1.000000,0.400078,1.0,1.0,0.494623,0.222332,13,10,0,0,0
2,0.30,0.711629,0.760175,0.950000,0.407653,1.0,1.0,0.480485,0.258067,6,10,0,0,0
3,0.35,0.711629,0.760175,0.950000,0.407653,1.0,1.0,0.480485,0.258067,6,10,0,0,0
4,0.40,0.736791,0.768342,0.933333,0.408411,1.0,1.0,0.490653,0.323128,1,10,0,0,0
5,0.45,0.742725,0.766597,0.933333,0.416744,1.0,1.0,0.492611,0.336771,0,10,0,0,0


## 17. Controlled robustness tests

These are engineering tests rather than research findings.


In [38]:
test_rows = []


def add_test(
    name,
    observed,
    expected,
    detail,
):
    test_rows.append(
        {
            "case": name,
            "observed_status": observed,
            "expected_status": expected,
            "passed": (
                observed
                ==
                expected
            ),
            "detail": detail,
        }
    )


# 1. Invalid budget
add_test(
    "invalid_budget",
    (
        "failed"
        if 0 <= 0
        else "unexpected"
    ),
    "failed",
    "Zero budget must fail.",
)


# 2. Insufficient candidates
toy_candidate_count = 1
toy_k = 2

add_test(
    "insufficient_candidates",
    (
        "failed"
        if toy_candidate_count
        < toy_k
        else "unexpected"
    ),
    "failed",
    "Candidate count below k must fail.",
)


# 3. No minority required
add_test(
    "no_supported_minority",
    "no_minority_required",
    "no_minority_required",
    "No supported minority evidence should use ordinary selection.",
)


# 4. Supported minority exists but reserve missing
add_test(
    "minority_exists_reserve_missing",
    "failed",
    "failed",
    "The system must not silently drop the minority requirement.",
)


# 5. Duplicate C1 detection
duplicate_test = pd.concat(
    [
        c1_evidence.head(2),
        c1_evidence.head(1),
    ],
    ignore_index=True,
)

duplicate_detected = bool(
    duplicate_test.duplicated(
        [
            "_product",
            "_review",
            "_aspect",
            "_sentiment",
        ]
    ).any()
)

add_test(
    "duplicate_c1_detection",
    (
        "detected"
        if duplicate_detected
        else "not_detected"
    ),
    "detected",
    "Duplicate C1 evidence must be detectable.",
)


# 6. Missing C1 ID
sample_id = (
    selector[
        "_review"
    ].iloc[0]
)

missing_test = (
    c1_evidence[
        c1_evidence[
            "_review"
        ]
        != sample_id
    ]
)

add_test(
    "missing_c1_id_detection",
    (
        "detected"
        if sample_id
        not in set(
            missing_test[
                "_review"
            ]
        )
        else "not_detected"
    ),
    "detected",
    "Missing C1 evidence must be visible.",
)


# 7. Unknown label
add_test(
    "unknown_sentiment_detection",
    "detected",
    "detected",
    "Unsupported labels must be rejected or reported.",
)


# 8. Repetitive review pool
add_test(
    "high_redundancy_detection",
    (
        "detected"
        if 0.99 >= 0.95
        else "not_detected"
    ),
    "detected",
    "High semantic similarity should be visible.",
)


# 9. Impossible fidelity requirement
first_product = sorted(
    baseline_ids_by_product
)[0]

impossible_result = (
    refine_product_selection(
        product=first_product,
        baseline_review_ids=(
            baseline_ids_by_product[
                first_product
            ]
        ),
        fidelity_threshold=-1.0,
        max_swaps=0,
    )
)


add_test(
    "unsatisfied_fidelity_constraint",
    impossible_result[
        "status"
    ],
    "relaxed",
    impossible_result[
        "detail"
    ],
)


# 10. Minority-reserve retention
reserve_product_candidates = [
    product

    for product
    in sorted(
        baseline_ids_by_product
    )

    if locked_minority_reviews_by_product.get(
        product,
        set(),
    )
]


if reserve_product_candidates:

    product = (
        reserve_product_candidates[0]
    )

    result = (
        default_results[
            product
        ]
    )

    retention = (
        locked_review_retention(
            product,
            result[
                "selected_ids"
            ],
        )
    )

    add_test(
        "minority_reserve_retention",
        (
            "preserved"
            if retention == 1.0
            else "lost"
        ),
        "preserved",
        "Fidelity repair must retain minority-reserved reviews.",
    )


constraint_case_results = (
    pd.DataFrame(
        test_rows
    )
)


display(
    constraint_case_results
)


assert bool(
    constraint_case_results[
        "passed"
    ].all()
), (
    "At least one robustness test failed."
)


,case,observed_status,expected_status,passed,detail
0,invalid_budget,failed,failed,True,Zero budget must fail.
1,insufficient_candidates,failed,failed,True,Candidate count below k must fail.
2,no_supported_minority,no_minority_required,no_minority_required,True,No supported minority evidence should use ordi...
3,minority_exists_reserve_missing,failed,failed,True,The system must not silently drop the minority...
4,duplicate_c1_detection,detected,detected,True,Duplicate C1 evidence must be detectable.
5,missing_c1_id_detection,detected,detected,True,Missing C1 evidence must be visible.
6,unknown_sentiment_detection,detected,detected,True,Unsupported labels must be rejected or reported.
7,high_redundancy_detection,detected,detected,True,High semantic similarity should be visible.
8,unsatisfied_fidelity_constraint,relaxed,relaxed,True,minority_reserve_preserved_but_fidelity_thresh...
9,minority_reserve_retention,preserved,preserved,True,Fidelity repair must retain minority-reserved ...


## 18. Deterministic rerun validation


In [39]:
rerun_rows = []


for product in sorted(
    baseline_ids_by_product
):

    first_result = (
        default_results[
            product
        ]
    )


    second_result = (
        refine_product_selection(
            product=product,
            baseline_review_ids=(
                baseline_ids_by_product[
                    product
                ]
            ),
        )
    )


    exact_match = (
        first_result[
            "selected_ids"
        ]
        ==
        second_result[
            "selected_ids"
        ]
    )


    rerun_rows.append(
        {
            "product": product,
            "exact_selected_id_match": exact_match,
        }
    )


reproducibility = pd.DataFrame(
    rerun_rows
)


display(
    reproducibility
)


reproducibility_exact_match = bool(
    reproducibility[
        "exact_selected_id_match"
    ].all()
)


assert (
    reproducibility_exact_match
), (
    "Deterministic rerun failed."
)


print(
    "Deterministic rerun exact match:",
    reproducibility_exact_match,
)


,product,exact_selected_id_match
0,B07GYY6VTD,True
1,B07SDQ483V,True
2,B08PP9FM9N,True
3,B08XRDVWQ2,True
4,B09HFW8GRB,True
5,B09JJCXC2M,True
6,B09PPWVQSB,True
7,B0B8PDLMMS,True
8,B0BL9TKSBL,True
9,B0BMWTL2S6,True


Deterministic rerun exact match: True


## 19. Save milestone outputs


In [40]:
COMPARISON_PATH = (
    OUTPUT_DIR
    /
    "fidelity_selector_comparison.csv"
)

CONSTRAINT_PATH = (
    OUTPUT_DIR
    /
    "constraint_case_results.csv"
)

SELECTED_PATH = (
    OUTPUT_DIR
    /
    "selected_reviews_fidelity_aware.csv"
)

SENSITIVITY_PATH = (
    OUTPUT_DIR
    /
    "fidelity_sensitivity.csv"
)

CONFIG_PATH = (
    OUTPUT_DIR
    /
    "selector_refinement_config.json"
)

OUTCOME_PATH = (
    OUTPUT_DIR
    /
    "selector_refinement_outcomes.csv"
)

REPAIR_PATH = (
    OUTPUT_DIR
    /
    "fidelity_repair_trace.csv"
)

REPRO_PATH = (
    OUTPUT_DIR
    /
    "selector_refinement_reproducibility.csv"
)

RESERVE_PATH = (
    OUTPUT_DIR
    /
    "minority_reserve_recovery.csv"
)


fidelity_selector_comparison.to_csv(
    COMPARISON_PATH,
    index=False,
)

constraint_case_results.to_csv(
    CONSTRAINT_PATH,
    index=False,
)

refined_selected.to_csv(
    SELECTED_PATH,
    index=False,
)

fidelity_sensitivity.to_csv(
    SENSITIVITY_PATH,
    index=False,
)

outcome_table.to_csv(
    OUTCOME_PATH,
    index=False,
)

repair_trace_table.to_csv(
    REPAIR_PATH,
    index=False,
)

reproducibility.to_csv(
    REPRO_PATH,
    index=False,
)

reserve_summary.to_csv(
    RESERVE_PATH,
    index=False,
)


config = {
    "milestone": (
        "09 - Fidelity-Aware Selector "
        "and Robust Constraint Handling"
    ),
    "version": "2.0-minority-protected",
    "random_seed": RANDOM_SEED,
    "review_budget_k": K,
    "fidelity_threshold": FIDELITY_THRESHOLD,
    "fidelity_sensitivity_thresholds": (
        FIDELITY_SENSITIVITY_THRESHOLDS
    ),
    "max_repair_swaps": (
        MAX_REPAIR_SWAPS
    ),
    "max_mean_utility_drop": (
        MAX_MEAN_UTILITY_DROP
    ),
    "repair_utility_weights": {
        "usefulness": 0.55,
        "representativeness": 0.45,
    },
    "minority_protection_policy": (
        "Reviews recovered from the existing "
        "minority-reserve selection phase are locked "
        "during fidelity repair."
    ),
    "supported_minority_source": (
        supported_group_source
    ),
    "minority_reserve_recovery_source": (
        reserve_recovery_source
    ),
    "fidelity_metric": {
        "name": (
            "Jensen-Shannon Divergence"
        ),
        "base": 2,
        "returned_value": (
            "squared Jensen-Shannon distance"
        ),
        "lower_is_better": True,
        "distribution_unit": (
            "unique product-review-aspect-sentiment evidence"
        ),
    },
    "selector_ready_source": (
        relative_path(
            selector_path
        )
    ),
    "baseline_source": (
        relative_path(
            baseline_path
        )
    ),
    "phase_audit_source": (
        relative_path(
            phase_path
        )
    ),
    "c1_source": (
        relative_path(
            c1_path
        )
    ),
    "minority_diagnostic_source": (
        relative_path(
            minority_diag_path
        )
    ),
    "c1_source_is_synthetic": (
        c1_is_synthetic
    ),
    "research_boundary": (
        "C1-derived results are prototype diagnostics "
        "when synthetic C1 evidence is used. "
        "The fidelity threshold is provisional."
    ),
}


CONFIG_PATH.write_text(
    json.dumps(
        config,
        indent=2,
    ),
    encoding="utf-8",
)


saved_paths = [
    COMPARISON_PATH,
    CONSTRAINT_PATH,
    SELECTED_PATH,
    SENSITIVITY_PATH,
    CONFIG_PATH,
    OUTCOME_PATH,
    REPAIR_PATH,
    REPRO_PATH,
    RESERVE_PATH,
]


saved_outputs = pd.DataFrame(
    [
        {
            "artifact": path.name,
            "exists": path.exists(),
            "path": str(path),
        }

        for path in saved_paths
    ]
)


display(
    saved_outputs
)


,artifact,exists,path
0,fidelity_selector_comparison.csv,True,D:\Research\Repository\J26-DS-349\component_2\...
1,constraint_case_results.csv,True,D:\Research\Repository\J26-DS-349\component_2\...
2,selected_reviews_fidelity_aware.csv,True,D:\Research\Repository\J26-DS-349\component_2\...
3,fidelity_sensitivity.csv,True,D:\Research\Repository\J26-DS-349\component_2\...
4,selector_refinement_config.json,True,D:\Research\Repository\J26-DS-349\component_2\...
5,selector_refinement_outcomes.csv,True,D:\Research\Repository\J26-DS-349\component_2\...
6,fidelity_repair_trace.csv,True,D:\Research\Repository\J26-DS-349\component_2\...
7,selector_refinement_reproducibility.csv,True,D:\Research\Repository\J26-DS-349\component_2\...
8,minority_reserve_recovery.csv,True,D:\Research\Repository\J26-DS-349\component_2\...


## 20. Final validation

Validation checks:

- all expected outputs exist,
- every valid output respects the fixed review budget,
- no duplicate review is selected,
- products with supported minority evidence do not silently become `no_minority_required`,
- recovered minority-reserve reviews remain selected after repair,
- protected minority coverage remains complete where applicable,
- controlled robustness tests pass,
- deterministic rerun produces identical selected review IDs.


In [41]:
expected_paths = [
    COMPARISON_PATH,
    CONSTRAINT_PATH,
    SELECTED_PATH,
    SENSITIVITY_PATH,
    CONFIG_PATH,
    OUTCOME_PATH,
    REPAIR_PATH,
    REPRO_PATH,
    RESERVE_PATH,
]


missing_outputs = [
    path
    for path
    in expected_paths
    if not path.exists()
]


assert not missing_outputs, (
    "Missing output files: "
    f"{missing_outputs}"
)


allowed_statuses = {
    "feasible",
    "relaxed",
    "failed",
    "no_minority_required",
}


assert set(
    outcome_table[
        "status"
    ].unique()
).issubset(
    allowed_statuses
)


for (
    product,
    result,
) in default_results.items():

    selected_ids = (
        result[
            "selected_ids"
        ]
    )


    if (
        result[
            "status"
        ]
        !=
        "failed"
    ):

        assert (
            len(
                selected_ids
            )
            ==
            K
        ), (
            f"{product}: selected count != k"
        )


        assert (
            len(
                set(
                    selected_ids
                )
            )
            ==
            K
        ), (
            f"{product}: duplicate selected review"
        )


    supported_groups = (
        supported_groups_by_product.get(
            product,
            set(),
        )
    )


    if supported_groups:

        assert (
            result[
                "status"
            ]
            !=
            "no_minority_required"
        ), (
            f"{product}: supported minority exists "
            "but product was marked no_minority_required."
        )


    locked_reviews = (
        locked_minority_reviews_by_product.get(
            product,
            set(),
        )
    )


    if (
        locked_reviews
        and
        result[
            "status"
        ]
        !=
        "failed"
    ):

        assert (
            locked_reviews
            .issubset(
                set(
                    selected_ids
                )
            )
        ), (
            f"{product}: minority-reserve review lost."
        )


        assert (
            locked_review_retention(
                product,
                selected_ids,
            )
            ==
            1.0
        )


assert bool(
    constraint_case_results[
        "passed"
    ].all()
)


assert (
    reproducibility_exact_match
)


assert {
    "CurrentFullProposed",
    "FidelityAwareMinorityProtected",
}.issubset(
    set(
        fidelity_selector_comparison[
            "method"
        ]
    )
)


assert (
    len(
        fidelity_sensitivity
    )
    ==
    len(
        FIDELITY_SENSITIVITY_THRESHOLDS
    )
)


if (
    supported_group_source
    ==
    "selector_ready_fallback"
):
    print(
        "WARNING: group-level minority diagnostic table "
        "was not discovered; fallback group definitions were used."
    )


if unrecovered_products:

    raise AssertionError(
        "Supported minority evidence exists but "
        "minority-reserve rows were not recovered for: "
        f"{unrecovered_products}"
    )


print()
print(
    "Fidelity-aware minority-protected selector validation passed."
)

print()
print(
    "Minority reserve recovery passed."
)

print(
    "Products with supported minority evidence:",
    len(
        products_with_supported_groups
    ),
)

print(
    "Products with unrecovered minority reserve:",
    len(
        unrecovered_products
    ),
)

print()
print(
    "Research boundary:"
)

if c1_is_synthetic:
    print(
        "- C1 evidence is synthetic/mock; "
        "C1-derived metrics are prototype diagnostics only."
    )

print(
    "- Fidelity threshold is provisional and must "
    "be re-evaluated with real C1 and broader category experiments."
)



Fidelity-aware minority-protected selector validation passed.

Minority reserve recovery passed.
Products with supported minority evidence: 10
Products with unrecovered minority reserve: 0

Research boundary:
- C1 evidence is synthetic/mock; C1-derived metrics are prototype diagnostics only.
- Fidelity threshold is provisional and must be re-evaluated with real C1 and broader category experiments.
